# Policy Assistant for XYZ Bank with Granite Switch and Mellea

Companion notebook to the blog post [One Model, Three Specialists](xyz-bank-policy-assistant.md).

**The story:** Priya is a compliance analyst at XYZ Bank. Branch staff ask her policy questions all day. She wants an assistant that answers from the bank's own documents and **refuses when it can't back an answer up**.

**What you'll build:** one assistant, running on one self-hosted Granite Switch model, that

1. checks the question for risk (`guardian`)
2. checks the policy documents can answer it (`rag.check_answerability`)
3. drafts a reply, then checks it against a reply rule (`core.requirement_check`)
4. logs a score for every step

The policy text and the XYZ Bank scenario are made up for illustration.

**Prerequisites:**

- A running vLLM server with a Granite Switch model (Step 2). This needs a GPU, so this notebook is not part of the Colab-tested recipes.
- Python 3.11 or 3.12
- Mellea 0.9 or newer, which includes the `core` and `guardian` intrinsics. Older versions (for example 0.3.0) fail on import.

## Background: the pieces

**Granite 4.1** is IBM's family of open (Apache 2.0) language models, in sizes such as 3B, 8B and 30B parameters. The 3B model has a 128K-token context window. On its own it is a general chat and text-generation model.

**Granite Switch** packs a Granite 4.1 base model and a set of small, independently trained *adapters* into **one checkpoint**. The 3B preview has 12, from three libraries: Core (requirement check, uncertainty, context attribution), RAG (query rewrite, answerability, hallucination detection, citations) and Guardian (safety, factuality, policy guardrails).

- The adapters are *activated LoRAs*: each only changes the model after a special control token, so what the base model already read stays valid.
- A small switch layer inside the model spots the control tokens and picks the adapter for each token. You don't write routing code.
- You select an adapter with `adapter_name` in the chat template, or let Mellea do it for you.

**Mellea** is a Python library for writing *generative programs*: structured, testable AI steps instead of long, fragile prompts. With Granite Switch, it turns each adapter into a plain function call such as `guardian.guardian_check(...)`.

> The 8B and 30B Switch releases bundle different adapters. IBM's Granite Switch 4.1 release post lists a Decision adapter, a Guardian adapter and Granite Speech 5.0 Turbo CTC. Check the model card of the size you pick.

## Step 1. Install relevant libraries

- **`mellea`**: provides the adapter functions and `OpenAIBackend`
- **`huggingface_hub`**: `load_embedded_adapters=True` uses it to fetch the adapter configs from Hugging Face
- **`openai`**: plain chat calls to the same server, with no adapter

In [ ]:
%pip install -q "mellea>=0.9" huggingface_hub openai

## Step 2. Serve the model

Run this in a separate terminal, on a machine with a GPU. Package requirements differ between the [model card](https://huggingface.co/ibm-granite/granite-switch-4.1-3b-preview) (older vLLM line) and the [`granite-switch` repo](https://github.com/generative-computing/granite-switch) (recent vLLM, PyTorch 2.11+ and CUDA 13+). Check both and pin the versions you use.

```bash
git clone https://github.com/generative-computing/granite-switch.git
cd granite-switch
pip install -e ".[vllm]"

python -m vllm.entrypoints.openai.api_server \
  --model ibm-granite/granite-switch-4.1-3b-preview --port 8000
```

> **Use the 3B preview for this notebook.** It bundles all 12 adapters, including the three used here. The 8B and 30B Switch models bundle different adapters (they bundle a Decision adapter and a Guardian adapter instead), so `check_answerability` and `requirement_check` won't work on them. If vLLM fails to size memory, add `--dtype bfloat16` and a smaller `--max-model-len`.

## Step 3. Connect Mellea to the server

`load_embedded_adapters=True` makes the model's built-in specialists available as Mellea functions.

In [ ]:
from openai import OpenAI
from mellea.backends.openai import OpenAIBackend
from mellea.formatters import TemplateFormatter
from mellea.stdlib.components import Document, Message
from mellea.stdlib.components.intrinsic import core, guardian, rag
from mellea.stdlib.context import ChatContext

MODEL = "ibm-granite/granite-switch-4.1-3b-preview"
URL = "http://localhost:8000/v1"

backend = OpenAIBackend(
    model_id=MODEL,
    formatter=TemplateFormatter(model_id=MODEL),
    base_url=URL,
    api_key="EMPTY",
    load_embedded_adapters=True,
)
llm = OpenAI(base_url=URL, api_key="EMPTY")  # plain chat, no adapter

## Step 4. The bank's policy documents and rules

Made-up excerpts. In real use, load your current approved documents.

In [ ]:
POLICY_TEXT = [
    "XYZ Bank KYC Policy 4.2: Customers rated high-risk must be re-verified every 2 years.",
    "XYZ Bank KYC Policy 4.3: Customers rated low-risk must be re-verified every 8 years.",
]
POLICY = [Document(t) for t in POLICY_TEXT]
RULE = "Quote the policy clause number and do not give legal advice."
THRESHOLD = 0.5  # tune this on your own data


def log(step, value):
    print(f"  [{step}] {value}")

## Step 5. The assistant

Each step is an ordinary function call. If a check fails, the question goes to a human instead of continuing.

> **Return values (Mellea 0.9):** `guardian_check` and `requirement_check` return a float between 0.0 and 1.0. For `guardian_check`, 1.0 means risk detected. `check_answerability` returns the string `"answerable"` or `"unanswerable"`, so it is compared as a string, not against the threshold.
>
> **How the repair loop works:** the writer is not shown `RULE` on the first try, so the rule check is a real test of the draft. If the check fails, the retry adds the rule to the prompt. If you put `RULE` in the first prompt, drafts will usually pass and the loop will never run.

In [ ]:
def ask(question: str) -> str:
    print(f"\nQ: {question}")

    # 1. Safety: is the request itself risky?
    risk = guardian.guardian_check(
        ChatContext().add(Message("user", question)),
        backend, criteria="harm", scoring_schema="user_prompt",
    )
    log("safety risk", f"{risk:.2f}")
    if risk >= THRESHOLD:
        return "Routed to a human reviewer (flagged by safety check)."

    # 2. Answerability: do our documents actually cover this?
    greeting = ChatContext().add(Message("assistant", "Hello, how can I help?"))
    answerable = rag.check_answerability(question, POLICY, greeting, backend)
    log("answerability", answerable)
    if answerable != "answerable":
        return "I can't answer this from the policy documents. Routed to Priya."

    # 3. Draft, then 4. check the draft against our reply rule (retry once).
    # The writer is NOT told RULE on the first try, so the check is a real test.
    # On the retry, the failed rule is added to the prompt (the repair step).
    sources = "\n".join(POLICY_TEXT)
    for attempt in range(2):
        system = f"Answer only from these policies and cite the clause number:\n{sources}"
        if attempt > 0:
            system += f"\nYour previous reply broke this rule. Follow it exactly: {RULE}"
        reply = llm.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": question},
            ],
        ).choices[0].message.content

        review = (ChatContext().add(Message("user", question))
                               .add(Message("assistant", reply)))
        score = core.requirement_check(review, backend, requirement=RULE)
        log(f"reply rule check (try {attempt + 1})", f"{score:.2f}")
        if score >= THRESHOLD:
            return reply
    return "Reply failed our rules twice. Routed to a human reviewer."

## Step 6. Four examples

### Example 1: The easy one

A high-risk customer question the documents cover. Expect low risk, answerable, a reply that quotes clause 4.2, and a passing rule check.

In [ ]:
print(ask("How often must a high-risk customer be re-verified?"))

### Example 2: "I don't know" is the right answer

The documents say nothing about trust accounts. Expect the answerability check to fail **before any answer is written**, so the question goes to Priya.

In [ ]:
print(ask("What is the re-verification rule for customers in a trust account?"))

### Example 3: A risky request

Expect the safety check to flag the request so it never reaches the writer.

In [ ]:
print(ask("How can I skip the KYC check for a friend's account?"))

### Example 4: The draft that breaks the rules

The first draft is written without being shown `RULE`, so a rule it can't guess will fail the check. Here `RULE` requires an opening phrase the writer was never told about. Expect a low `reply rule check` score on try 1, then a second draft (now told the rule) that passes. If it fails twice, the question escalates. Re-run Step 4 to restore the original `RULE`.

In [ ]:
RULE = "Start the reply with the words 'Per XYZ Bank policy,' and quote the clause number."
print(ask("How often must a low-risk customer be re-verified?"))

## What this does not do

- It's a **pre-screen, not an approval.** A person still owns the decision, and the thresholds need tuning on your own data.
- The policy text is made up, and the expected behaviour above is not captured output. Run it on your own documents and test questions before you trust it.